In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# =========================
# 1. Paths
# =========================

features_path = r"C:\Users\eduar\Documents\GitHub\2026-PDS-Xingxiulong\data\Separate csv of features\features_borders.csv"

# Change this to the real path of your asymmetry file
asymmetry_path = r"C:\Users\eduar\Documents\GitHub\2026-PDS-Xingxiulong\data\Separate csv of features\features_asymmetry.csv"

# =========================
# 2. Read data
# =========================

features_df = pd.read_csv(features_path)
asymmetry_df = pd.read_csv(asymmetry_path)

features_df.columns = features_df.columns.str.strip()
asymmetry_df.columns = asymmetry_df.columns.str.strip()

features_df["diagnostic"] = features_df["diagnostic"].astype(str).str.strip().str.upper()
features_df["mask_name"] = features_df["mask_name"].astype(str).str.strip()
asymmetry_df["filename"] = asymmetry_df["filename"].astype(str).str.strip()

# =========================
# 3. Merge asymmetry score into features data
# =========================

df = features_df.merge(
    asymmetry_df[["filename", "asymmetry_score"]],
    left_on="mask_name",
    right_on="filename",
    how="left"
)

# Check if merge worked
print("Rows in final dataset:", len(df))
print("Missing asymmetry scores:", df["asymmetry_score"].isna().sum())

# =========================
# 4. Create cancer variable
# =========================

cancerous = ["BCC", "SCC", "MEL"]
non_cancerous = ["ACK", "NEV", "SEK"]

df_model = df[df["diagnostic"].isin(cancerous + non_cancerous)].copy()

df_model["cancer"] = df_model["diagnostic"].isin(cancerous).astype(int)

# =========================
# 5. Select features
# =========================

features = [
    "compactness",
    "asymmetry_score"
]

df_model = df_model.dropna(subset=features + ["cancer"])

X = df_model[features]
y = df_model["cancer"]

print("\nCancer distribution:")
print(y.value_counts())

print("\nDiagnostic distribution:")
print(df_model["diagnostic"].value_counts())

# =========================
# 6. Logistic regression model
# =========================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression())
])

# =========================
# 7. Cross-validation AUC
# =========================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("\nAUC scores:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Standard deviation:", auc_scores.std())

# =========================
# 8. Train-test split evaluation
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("\nTest AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================
# 9. Coefficients
# =========================

logistic_model = model.named_steps["logistic"]

coef_df = pd.DataFrame({
    "feature": features,
    "coefficient": logistic_model.coef_[0],
    "odds_ratio": np.exp(logistic_model.coef_[0])
})

print("\nModel coefficients:")
print(coef_df)

Rows in final dataset: 2100
Missing asymmetry scores: 0

Cancer distribution:
cancer
0    1052
1    1048
Name: count, dtype: int64

Diagnostic distribution:
diagnostic
BCC    815
ACK    631
NEV    220
SEK    201
SCC    184
MEL     49
Name: count, dtype: int64

AUC scores: [0.55243883 0.57994512 0.55920635 0.57176871 0.53548753]
Mean AUC: 0.5597693078637382
Standard deviation: 0.015457727902574275

Test AUC: 0.5868284329376252

Confusion Matrix:
[[108 155]
 [ 83 179]]

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.41      0.48       263
           1       0.54      0.68      0.60       262

    accuracy                           0.55       525
   macro avg       0.55      0.55      0.54       525
weighted avg       0.55      0.55      0.54       525


Model coefficients:
           feature  coefficient  odds_ratio
0      compactness     0.111405    1.117848
1  asymmetry_score    -0.350284    0.704488


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X = df_model[["compactness","asymmetry_score"]]
y = df_model["cancer"]

model = LogisticRegression()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("AUC scores:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Standard deviation:", auc_scores.std())
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

AUC scores: [0.55207601 0.58033062 0.55845805 0.57131519 0.5355102 ]
Mean AUC: 0.5595380154495716
Standard deviation: 0.015525577969980903
[[108 155]
 [ 83 179]]
              precision    recall  f1-score   support

           0       0.57      0.41      0.48       263
           1       0.54      0.68      0.60       262

    accuracy                           0.55       525
   macro avg       0.55      0.55      0.54       525
weighted avg       0.55      0.55      0.54       525

